# 2.4 · Patterns: cancellation, progress, pagination and logging

**通用模式：取消、进度、分页与日志**

> **After this lesson you can:** Implement and consume progress notifications, cancellation with timeouts, cursor-based pagination, and per-request logging, and explain what to use now that logging is deprecated.

Official reading: [Spec: Patterns overview](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/index) · [Spec: Cancellation](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/cancellation) · [Spec: Progress](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/progress) · [Spec: Pagination](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/pagination) · [Spec: Logging](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/logging)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. Progress, seen through the SDK
`ctx.report_progress(progress, total, message)` sends `notifications/progress`, but **only** if the request carried a `progressToken`. `Client.call_tool(..., progress_callback=...)` creates the token for you and routes the notifications to your callback.

In [ ]:
import json
import os
import sys
import tempfile
import textwrap
import time
import warnings
from pathlib import Path

import anyio
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from mcp.server.mcpserver import Context
from mcp.shared.exceptions import MCPDeprecationWarning, MCPError

indexer = MCPServer("indexer")


@indexer.tool()
async def index_repo(files: int, ctx: Context) -> str:
    """Index a repository, reporting progress per file."""
    for i in range(1, files + 1):
        await anyio.sleep(0.05)
        await ctx.report_progress(i, files, f"indexed file {i}/{files}")
    return f"indexed {files} files"


async def show_progress(progress, total, message):
    bar = "#" * int(20 * progress / total)
    print(f"  [{bar:<20}] {progress:>4.0f}/{total:.0f}  {message}")


async with Client(indexer) as client:
    result = await client.call_tool("index_repo", {"files": 5}, progress_callback=show_progress)
    print(result.content[0].text)
    quiet = await client.call_tool("index_repo", {"files": 5})        # no callback -> no token -> no notifications
    print(quiet.content[0].text, "(no progressToken, so no notifications)")

### 2. Progress on the wire, and a client that checks the rules
Over raw stdio you can see the token travel. We also run every notification through a small validator for the spec's rules: the token must be known and active, and `progress` must strictly increase. A deliberately buggy tool goes backwards.

In [ ]:
import queue
import subprocess
import threading

workdir = Path(tempfile.mkdtemp())
(workdir / "progress_server.py").write_text(textwrap.dedent('''
    import anyio
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    mcp = MCPServer("progress-lab", log_level="ERROR")

    @mcp.tool()
    async def good(ctx: Context) -> str:
        """Well-behaved progress."""
        for i in (0.25, 0.5, 1.0):
            await ctx.report_progress(i, 1.0, f"{i:.0%}")
        return "ok"

    @mcp.tool()
    async def buggy(ctx: Context) -> str:
        """Progress that goes backwards."""
        for i in (30, 20, 40):
            await ctx.report_progress(i)
        return "ok"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
ENVELOPE = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
            "io.modelcontextprotocol/clientCapabilities": {}}


def start(script, env=None):
    proc = subprocess.Popen([sys.executable, str(workdir / script)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                            stderr=subprocess.DEVNULL, text=True, env=env)
    inbox = queue.Queue()
    threading.Thread(target=lambda: [inbox.put(l) for l in proc.stdout], daemon=True).start()
    return proc, inbox


def send(proc, message):
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()


class ProgressTracker:
    def __init__(self):
        self.active = {}                                     # token -> last progress value

    def start(self, token):
        self.active[token] = None

    def check(self, params):
        token, value = params["progressToken"], params["progress"]
        if token not in self.active:
            return "VIOLATION: unknown or finished token"
        last = self.active[token]
        self.active[token] = value
        if last is not None and value <= last:
            return f"VIOLATION: progress went {last} -> {value} (MUST increase)"
        return "ok"

    def finish(self, token):
        self.active.pop(token, None)


proc, inbox = start("progress_server.py")
tracker = ProgressTracker()
for id_, tool in [(1, "good"), (2, "buggy")]:
    token = f"tok-{tool}"
    tracker.start(token)
    send(proc, {"jsonrpc": "2.0", "id": id_, "method": "tools/call",
                "params": {"name": tool, "arguments": {}, "_meta": {**ENVELOPE, "progressToken": token}}})
    while True:
        msg = json.loads(inbox.get(timeout=10))
        if msg.get("method") == "notifications/progress":
            print(f"  {json.dumps(msg['params']):<85} {tracker.check(msg['params'])}")
        else:
            tracker.finish(token)
            print(f"  response id={msg['id']} -> token {token!r} retired\n")
            break
proc.stdin.close()
_ = proc.wait(timeout=10)

### 3. Cancellation from the SDK: give up, time out, and what the server sees
The server appends to an events file so we can watch it from outside. When the caller's `anyio` scope gives up, the SDK sends `notifications/cancelled` on stdio and the handler is interrupted. `read_timeout_seconds` does the same and raises an error.

In [ ]:
(workdir / "cancel_server.py").write_text(textwrap.dedent('''
    import os
    import anyio
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    mcp = MCPServer("cancel-lab", log_level="ERROR")

    def note(text):
        with open(os.environ["EVENTS_FILE"], "a") as f:
            f.write(text + "\\n")

    @mcp.tool()
    async def crunch(seconds: float, ctx: Context) -> str:
        """A long job."""
        note(f"request {ctx.request_id}: started ({seconds}s of work)")
        try:
            await anyio.sleep(seconds)
        except BaseException as exc:
            note(f"request {ctx.request_id}: interrupted by {type(exc).__name__}, resources freed")
            raise
        note(f"request {ctx.request_id}: finished")
        return "crunched"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
events_file = workdir / "events.log"
events_file.write_text("")
env = {**os.environ, "EVENTS_FILE": str(events_file)}
params = StdioServerParameters(command=sys.executable, args=[str(workdir / "cancel_server.py")], env=env)

async with Client(params) as client:
    with anyio.move_on_after(0.5) as scope:                 # the user pressed "stop" after 0.5 s
        await client.call_tool("crunch", {"seconds": 5})
    print("caller gave up:", scope.cancelled_caught)

    try:
        await client.call_tool("crunch", {"seconds": 5}, read_timeout_seconds=0.5)
    except MCPError as err:
        print(f"timeout -> MCPError {err.error.code}: {err.error.message}")

    print((await client.call_tool("crunch", {"seconds": 0.1})).content[0].text)
    await anyio.sleep(0.2)

print("\nserver-side events:")
print(events_file.read_text())

The timeout surfaces as code `-32001`, from the **legacy implementation-defined** range. The spec assigns no code to purely local errors, so the SDK has to pick one. Remember that it was raised by your own client, not sent by the server.

### 4. Cancellation by hand, including the races
Raw stdio again. We cancel a running request, then send two *invalid* cancellations (an already-finished id and an unknown id). The spec says the server should silently ignore those, and the connection keeps working.

In [ ]:
proc, inbox = start("cancel_server.py", env=env)


def next_message(timeout=2.0):
    try:
        return json.loads(inbox.get(timeout=timeout))
    except queue.Empty:
        return None


send(proc, {"jsonrpc": "2.0", "id": "job-1", "method": "tools/call",
            "params": {"name": "crunch", "arguments": {"seconds": 10}, "_meta": ENVELOPE}})
time.sleep(1.0)                                              # let it start (process start-up + work)
send(proc, {"jsonrpc": "2.0", "method": "notifications/cancelled",
            "params": {"requestId": "job-1", "reason": "user pressed stop"}})
print("reply to job-1 after cancel:", next_message())

send(proc, {"jsonrpc": "2.0", "id": "job-2", "method": "tools/call",
            "params": {"name": "crunch", "arguments": {"seconds": 0.05}, "_meta": ENVELOPE}})
print("job-2:", next_message(5)["result"]["content"][0]["text"])
send(proc, {"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": "job-2"}})   # too late
send(proc, {"jsonrpc": "2.0", "method": "notifications/cancelled", "params": {"requestId": "never-sent"}})
print("after two invalid cancels:", next_message(1.0), "(silently ignored)")

send(proc, {"jsonrpc": "2.0", "id": "job-3", "method": "tools/call",
            "params": {"name": "crunch", "arguments": {"seconds": 0.05}, "_meta": ENVELOPE}})
print("job-3 still works:", next_message(5)["result"]["content"][0]["text"])
proc.stdin.close()
_ = proc.wait(timeout=10)
print("\nserver-side events:\n" + "\n".join(l for l in events_file.read_text().splitlines() if "job-" in l))

### 5. A watchdog: idle timeout reset by progress, plus a hard maximum
This implements the spec's timeout advice. Progress extends the *idle* deadline, but a hard cap ends even a request that reports progress forever.

In [ ]:
forever = MCPServer("forever")


@forever.tool()
async def stream_forever(ctx: Context) -> str:
    """Reports progress endlessly (a misbehaving server)."""
    i = 0
    while True:
        i += 1
        await ctx.report_progress(i, message=f"tick {i}")
        await anyio.sleep(0.2)


async def call_with_watchdog(client, name, args, idle=0.5, hard_max=1.5):
    started = last_progress = time.monotonic()

    async def on_progress(progress, total, message):
        nonlocal last_progress
        last_progress = time.monotonic()

    async with anyio.create_task_group() as tg:
        outcome = {}

        async def run():
            outcome["result"] = (await client.call_tool(name, args, progress_callback=on_progress)).content[0].text
            tg.cancel_scope.cancel()

        tg.start_soon(run)
        while True:
            await anyio.sleep(0.05)
            now = time.monotonic()
            if now - last_progress > idle:
                outcome["error"] = f"idle timeout ({idle}s without progress)"
            elif now - started > hard_max:
                outcome["error"] = f"hard maximum of {hard_max}s reached despite progress"
            if outcome:
                tg.cancel_scope.cancel()             # cancelling the call makes the SDK cancel the request
                break
    return outcome


async with Client(forever) as client:
    print(await call_with_watchdog(client, "stream_forever", {}))
async with Client(indexer) as client:
    print(await call_with_watchdog(client, "index_repo", {"files": 8}, idle=0.5, hard_max=5))

### 6. Pagination with opaque cursors
`MCPServer` returns every tool in one page. To paginate, use the low-level `Server` and write the `tools/list` handler yourself. The cursor here is a base64 blob the client must not interpret, and a bad cursor gets `-32602`.

In [ ]:
import base64

import mcp_types as types
from mcp.server import Server

CATALOG = [types.Tool(name=f"report_{i:02d}", description=f"Report number {i}", input_schema={"type": "object"})
           for i in range(7)]
PAGE_SIZE = 3                                                   # the SERVER decides


def make_cursor(offset: int) -> str:
    return base64.urlsafe_b64encode(json.dumps({"o": offset}).encode()).decode()


async def list_tools(ctx, params: types.PaginatedRequestParams | None) -> types.ListToolsResult:
    offset = 0
    if params is not None and params.cursor is not None:
        try:
            offset = json.loads(base64.urlsafe_b64decode(params.cursor))["o"]
        except Exception:
            raise MCPError(code=types.INVALID_PARAMS, message="Invalid cursor")
    page = CATALOG[offset:offset + PAGE_SIZE]
    more = offset + PAGE_SIZE < len(CATALOG)
    return types.ListToolsResult(tools=page, next_cursor=make_cursor(offset + PAGE_SIZE) if more else None)


pager = Server("pager", on_list_tools=list_tools)

async with Client(pager) as client:
    first = await client.list_tools()
    show = first.model_dump(by_alias=True, exclude_none=True, mode="json")
    print("first page on the wire:", json.dumps({k: v for k, v in show.items() if k != "tools"}),
          "| tools:", [t["name"] for t in show["tools"]])

    everything, cursor, pages = [], None, 0
    while True:                                                  # the loop every client should have
        page = await client.list_tools(cursor=cursor)
        pages += 1
        everything += [t.name for t in page.tools]
        if page.next_cursor is None:                             # only a MISSING cursor ends the loop
            break
        cursor = page.next_cursor
    print(f"collected {len(everything)} tools in {pages} pages:", everything)

    try:
        await client.list_tools(cursor="not-a-real-cursor")
    except MCPError as err:
        print("invalid cursor ->", err.error.code, err.error.message)

### 7. Logging, deprecated but still on the wire
A client opts in **per request** by setting `io.modelcontextprotocol/logLevel`. The SDK's `Client(log_level=..., logging_callback=...)` stamps that key on every request. With no level, the server must stay silent. The SDK marks the whole feature with `MCPDeprecationWarning`.

In [ ]:
chatty = MCPServer("chatty")


@chatty.tool()
async def migrate(ctx: Context) -> str:
    """Run a migration, logging along the way."""
    await ctx.debug("opening connection pool")
    await ctx.info({"step": "migrate", "tables": 12})
    await ctx.warning("table 'legacy_users' has no primary key")
    return "migrated"


async def print_log(params):
    print(f"   log [{params.level:<7}] {params.data}")


with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", MCPDeprecationWarning)
    for level in ["debug", "warning", None]:
        print(f"log_level={level!r}:")
        async with Client(chatty, log_level=level, logging_callback=print_log) as client:
            print("  ", (await client.call_tool("migrate", {})).content[0].text)
print("\ndeprecation warnings raised:", sorted({str(w.message) for w in caught}))

On the raw wire, the level is just a `_meta` key, and an unknown level is rejected:

In [ ]:
(workdir / "log_server.py").write_text(textwrap.dedent('''
    import warnings
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context
    warnings.simplefilter("ignore")
    mcp = MCPServer("log-lab", log_level="ERROR")

    @mcp.tool()
    async def migrate(ctx: Context) -> str:
        """Run a migration."""
        await ctx.info("migration started")
        return "migrated"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
proc, inbox = start("log_server.py")
for id_, level in [(1, "info"), (2, "loud")]:
    send(proc, {"jsonrpc": "2.0", "id": id_, "method": "tools/call",
                "params": {"name": "migrate", "arguments": {},
                           "_meta": {**ENVELOPE, "io.modelcontextprotocol/logLevel": level}}})
    while True:
        msg = json.loads(inbox.get(timeout=10))
        print(f"logLevel={level!r}:", json.dumps(msg)[:150])
        if "id" in msg:
            break
send(proc, {"jsonrpc": "2.0", "id": 3, "method": "server/discover", "params": {"_meta": ENVELOPE}})
caps = json.loads(inbox.get(timeout=10))["result"]["capabilities"]
print("\ndeclared capabilities:", sorted(caps), "(the spec says log emitters MUST declare 'logging')")
proc.stdin.close()
_ = proc.wait(timeout=10)

Notice the last line: this SDK version emits `notifications/message` but does not advertise the `logging` capability in `server/discover`. Treat capability lists as hints and handle what actually arrives.

### 8. The replacement: structured logs on stderr
For stdio servers, the migration path is simply stderr. Here the server writes one JSON object per line to stderr, and the host captures it separately from the protocol stream. No opt-in or capability is needed, and nothing reaches the model's context.

In [ ]:
(workdir / "stderr_server.py").write_text(textwrap.dedent('''
    import json, logging, sys, time
    from mcp.server import MCPServer

    class JsonLines(logging.Formatter):
        def format(self, record):
            return json.dumps({"ts": round(time.time(), 3), "level": record.levelname.lower(),
                               "logger": record.name, "msg": record.getMessage()})

    handler = logging.StreamHandler(sys.stderr)          # never stdout on stdio!
    handler.setFormatter(JsonLines())
    log = logging.getLogger("billing")
    log.addHandler(handler)
    log.setLevel(logging.INFO)
    log.propagate = False

    mcp = MCPServer("billing", log_level="ERROR")

    @mcp.tool()
    def charge(cents: int) -> str:
        """Charge a (pretend) card."""
        log.info("charge requested: %d cents", cents)
        if cents > 10_000:
            log.warning("large charge flagged for review")
        return "charged"

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''))
proc = subprocess.Popen([sys.executable, str(workdir / "stderr_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
send(proc, {"jsonrpc": "2.0", "id": 1, "method": "tools/call",
            "params": {"name": "charge", "arguments": {"cents": 25_000}, "_meta": ENVELOPE}})
print("stdout (protocol):", proc.stdout.readline().strip()[:110], "...")
proc.stdin.close()
_ = proc.wait(timeout=10)
print("stderr (logs the host can capture):")
for line in proc.stderr.read().splitlines():
    if line.startswith("{"):
        print("  ", json.loads(line))

For remote servers, the equivalent is OpenTelemetry. The client puts `traceparent` in `_meta` (lesson 2.1), and the server exports spans and logs to your observability backend, correlated with the agent's trace.

### Your turn
1. Make `index_repo` report progress with `total=None` (unknown size). Update `show_progress` so it still renders something useful. Which field must still increase?
2. Change `call_with_watchdog` so that on a timeout it also prints how many progress notifications were received. Then try `idle=0.1` on `index_repo`: what goes wrong, and how would you pick the idle value in production?
3. Make the pager return `next_cursor=""` for the last page instead of `None`. Does the client loop in section 6 terminate? What does the spec say a client must do with `""`, and who has the bug?
4. Rewrite the `migrate` tool in section 7 to log to stderr as in section 8, and list two kinds of data that must never appear in either kind of log.